# 06. 공간 군집 강건성 — 이웃 셀끼리 비슷해서 p값이 부풀려진 건 아닌가

**문제**: 20분 이내 셀 11개는 GTX역 3곳 주변에 몰려 있다. 가까운 셀끼리는 오차가 같이 움직일 수 있는데(공간자기상관), 지금까지의 표준오차(HC1)는 셀이 서로 독립이라고 가정한다 → p값이 실제보다 작게 나왔을 수 있다.

**확인 방법**
1. 잔차의 공간자기상관 검정 (**Moran's I**, 1km 이내 이웃, 순열검정 999회)
2. **읍면동 군집 표준오차** (41개 읍면동으로 묶음)
3. **공간 HAC(Conley) 표준오차**: 거리 1·2·5km 이내 셀끼리의 오차 상관을 허용
4. **최근접 역 고정효과**: 운정중앙·킨텍스·대곡 권역 '안에서' 비교 (권역 간 차이를 제거)

대상 모형: 03의 보행시간 구간 더미 변화율 모형 (Poisson, offset = log 2024년 승차)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm

data = load_model_data()  # 1_받은코드_정리/output/model_dataset.csv
X_DUMMY = ["walk_20", "walk_20_60", "ic_access", "competing_share", "pop_20_50", "buildings"]

In [2]:
# 읍면동 (IC 파일에 있는 이름. 한글 인코딩이 깨져 있어 복원)
def fix_korean(text):
    try:
        return str(text).encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return str(text)

emd = read_csv(IC_PATH)[["셀 ID", "emd_nm_k"]].assign(읍면동=lambda d: d.emd_nm_k.map(fix_korean))
data = data.merge(emd[["셀 ID", "읍면동"]], on="셀 ID", how="left")
centers = stable_centers().set_index("셀 ID")
print(f"읍면동 {data.읍면동.nunique()}개 | 20분 이내 셀의 읍면동: {data.loc[data.walk_20 == 1, '읍면동'].value_counts().to_dict()}")

SPECS = {"오전": ("board_2025", "board_2024"), "종일": ("board_2025_all", "board_2024_all")}


def fit(label):
    after, before = SPECS[label]
    frame = data[data[before] > 0].reset_index(drop=True)
    X = sm.add_constant(frame[X_DUMMY])
    result = sm.GLM(frame[after], X, family=sm.families.Poisson(), offset=np.log(frame[before])).fit()
    return frame, X, result


def distance_matrix(frame):
    xy = np.c_[centers.loc[frame["셀 ID"]].geometry.x, centers.loc[frame["셀 ID"]].geometry.y]
    return np.sqrt(((xy[:, None] - xy[None]) ** 2).sum(-1))  # m

읍면동 41개 | 20분 이내 셀의 읍면동: {'목동동': 4, '동패동': 2, '대화동': 2, '화정동': 1, '주엽동': 1, '장항동': 1}


## 1. 잔차의 공간자기상관 (Moran's I)

In [3]:
def morans_i(values, weights):
    z = values - values.mean()
    return len(z) / weights.sum() * (z @ weights @ z) / (z @ z)


rng = np.random.default_rng(0)
rows = []
for label in SPECS:
    frame, X, result = fit(label)
    D = distance_matrix(frame)
    W = ((D > 0) & (D <= 1000)).astype(float)
    W[W.sum(1) > 0] /= W.sum(1)[W.sum(1) > 0][:, None]
    residual = np.asarray(result.resid_pearson)
    observed = morans_i(residual, W)
    simulated = np.array([morans_i(rng.permutation(residual), W) for _ in range(999)])
    rows.append({"모형": label, "Moran's I": observed, "순열 p": (np.sum(simulated >= observed) + 1) / 1000,
                 "셀당 평균 이웃 수(1km)": ((D > 0) & (D <= 1000)).sum(1).mean()})
pd.DataFrame(rows).round(3)

,모형,Moran's I,순열 p,셀당 평균 이웃 수(1km)
0,오전,0.254,0.004,3.525
1,종일,-0.077,0.823,3.652


## 2~4. 표준오차를 바꿔도 결과가 유지되나

In [4]:
def conley_p(frame, X, result, cutoff_m):
    """Conley 공간 HAC (Bartlett 커널): cutoff 이내 셀끼리의 점수 상관을 허용."""
    Xv, mu = X.values, np.asarray(result.mu)
    score = Xv * (frame[SPECS[label][0]].values - mu)[:, None]
    bread = np.linalg.inv(Xv.T @ (Xv * mu[:, None]))
    kernel = np.clip(1 - distance_matrix(frame) / cutoff_m, 0, None)
    se = np.sqrt(np.diag(bread @ (score.T @ kernel @ score) @ bread))
    return pd.Series(2 * norm.sf(np.abs(result.params.values / se)), index=X.columns)


table = {}
for label in SPECS:
    frame, X, result = fit(label)
    after, before = SPECS[label]
    rows = {"계수 (변화율 차이 %)": (np.exp(result.params) - 1) * 100,
            "p: HC1 (기존)": sm.GLM(frame[after], X, family=sm.families.Poisson(), offset=np.log(frame[before])).fit(cov_type="HC1").pvalues,
            "p: 읍면동 군집": sm.GLM(frame[after], X, family=sm.families.Poisson(), offset=np.log(frame[before])).fit(
                cov_type="cluster", cov_kwds={"groups": pd.factorize(frame["읍면동"])[0]}).pvalues}
    for km in (1, 2, 5):
        rows[f"p: Conley {km}km"] = conley_p(frame, X, result, km * 1000)
    table[label] = pd.DataFrame(rows).loc[["walk_20", "walk_20_60", "ic_access"]].rename(index=VARIABLE_LABELS)
pd.concat(table, axis=0).round(4)

계수 (변화율 차이 %)  p: HC1 (기존)  p: 읍면동 군집  p: Conley 1km  p: Conley 2km  p: Conley 5km
오전 보행 20분 이내           -42.3252       0.0034     0.0131         0.0025         0.0043         0.0085
   보행 20~60분           -22.4820       0.1509     0.2431         0.1551         0.1796         0.2075
   IC까지 도로거리(km)        -8.8156       0.0039     0.0467         0.0088         0.0211         0.0363
종일 보행 20분 이내           -50.4714       0.0000     0.0000         0.0000         0.0000         0.0000
   보행 20~60분           -28.8465       0.0134     0.0092         0.0068         0.0052         0.0078
   IC까지 도로거리(km)        -5.1690       0.0248     0.0246         0.0265         0.0198         0.0108

In [5]:
# 4. 최근접 역 고정효과: 같은 역 권역 안에서만 비교
rows = []
for label, (after, before) in SPECS.items():
    frame = data[data[before] > 0]
    X = pd.concat([frame[X_DUMMY], pd.get_dummies(frame.nearest_station, prefix="역", drop_first=True, dtype=float)], axis=1)
    result = poisson_rate(frame[after], frame[before], X, cluster=frame["읍면동"])
    for name in ["walk_20", "walk_20_60"]:
        rows.append({"모형": label, "변수": VARIABLE_LABELS[name], "변화율 차이(%)": (np.exp(result.params[name]) - 1) * 100, "p (읍면동 군집)": result.pvalues[name]})
pd.DataFrame(rows).round(4)

,모형,변수,변화율 차이(%),p (읍면동 군집)
0,오전,보행 20분 이내,-31.4889,0.0376
1,오전,보행 20~60분,-21.1326,0.2019
2,종일,보행 20분 이내,-52.2872,0.0002
3,종일,보행 20~60분,-31.8485,0.0035


## 결론
- **오전 잔차에는 공간자기상관이 있다** (Moran's I = 0.25, p = 0.004). 그래서 기존 HC1 p값은 낙관적일 수 있었다. 종일 잔차에는 없다.
- 그래도 **20분 이내 효과는 모든 표준오차에서 유의하다**:
  - 오전: 읍면동 군집 p = 0.013, Conley 1~5km p ≤ 0.009
  - 종일: 모두 p < 0.001
- **역 고정효과**(같은 역 권역 안에서만 비교)를 넣어도 20분 이내 효과가 유지된다: 오전 −31% (p = 0.04), 종일 −52% (p < 0.001).
- 20~60분 효과는 종일에서만 유의하다(모든 표준오차에서).
- IC까지 도로거리(−, IC에서 멀수록 더 감소)는 모든 표준오차에서 p < 0.05다.
- 참고: 역 3곳 단위 군집은 군집 수가 너무 적어서(3개) 쓰지 않았다.